# MDRS-Net++: Explainable Multi-Disease Risk Stratification Network

## Novelty Statement

This work proposes **MDRS-Net++**, a single joint neural architecture for multi-disease risk stratification that goes beyond training separate classifiers per disease or reusing off-the-shelf ensemble models. The architecture contributes four coupled innovations, evaluated jointly and in isolation via ablation:

1. **Disease-adaptive Feature Interaction Gating (DFIG)** — a learned, per-feature, magnitude-aware sigmoid gate `g = sigmoid(|x| * w)` combined with a residual pathway. Unlike a static feature-selection step, DFIG adapts per disease branch during training and its learned weights `w` double as a **built-in, architecture-native explainability signal** (see Explainable Risk Stratification layer below) — the model does not need a post-hoc SHAP/LIME wrapper to justify its own predictions.
2. **Temporal Risk Encoding (TRE)** — a sinusoidal, age-conditioned positional encoding (inspired by transformer positional embeddings but re-purposed for physiological aging) with an exponential recency-decay term, injected as a second modality alongside the raw clinical features.
3. **Cross-Modal Attention Fusion** — a lightweight two-token cross-attention block that fuses the DFIG-gated feature modality with the TRE age modality, replacing naive feature concatenation with a learned, disease-specific fusion.
4. **Shared Trunk with Disease-Specific Heads** — a shared representation trunk across all diseases enables limited positive transfer across heterogeneous disease datasets, while disease-specific output heads preserve per-disease calibration.

On top of the architecture, we add an **Explainable Risk Stratification (XRS) layer** that converts raw probabilities into four clinically interpretable risk tiers (Low / Moderate / High / Critical) and produces per-patient, per-feature attribution scores derived directly from the DFIG gates — turning the model's own internal computation into the explanation, rather than approximating it externally.

**What distinguishes this from "integration of existing techniques":** DFIG, TRE, and the disease-specific Cross-Modal Attention Fusion are custom-designed modules (implemented here from first principles in NumPy, including full forward/backward passes), not calls to pre-built attention/embedding libraries. The ablation study below isolates the marginal contribution of each module, and statistical significance testing (Wilcoxon signed-rank + Bonferroni correction) is used to support any claim of improvement, rather than reporting a single accuracy number.


## System Architecture

```
                         ┌────────────────────────────────────────────────┐
                         │                 Per-Disease Input               │
                         │   (CVD | Diabetes | COPD | Alzheimer's |        │
                         │    Liver | Kidney | ICU Critical-Illness)       │
                         └───────────────────────┬──────────────────────────┘
                                                  ▼
                    ┌─────────────────────────────────────────────────┐
                    │            DISEASE BRANCH (per disease)          │
                    │                                                   │
                    │   raw features ──► DFIG (feature gate) ──► mod_a │
                    │   age feature   ──► TRE (age encoding)  ──► mod_b │
                    │                                                   │
                    │        mod_a, mod_b ──► Cross-Modal Attention     │
                    │                       Fusion ──► fused vector     │
                    └───────────────────────┬───────────────────────────┘
                                             ▼
                    ┌─────────────────────────────────────────────────┐
                    │      SHARED TRUNK (2-layer MLP, dropout)         │
                    │      (positive transfer across diseases)         │
                    └───────────────────────┬───────────────────────────┘
                                             ▼
                    ┌─────────────────────────────────────────────────┐
                    │        DISEASE-SPECIFIC OUTPUT HEAD              │
                    │        → risk logit → sigmoid → probability      │
                    └───────────────────────┬───────────────────────────┘
                                             ▼
                    ┌─────────────────────────────────────────────────┐
                    │     EXPLAINABLE RISK STRATIFICATION (XRS)         │
                    │  • Risk tiers: Low / Moderate / High / Critical  │
                    │  • Per-feature attribution from DFIG gates       │
                    └───────────────────────────────────────────────────┘
```

**Validation loop wrapped around the architecture:** stratified 5-fold cross-validation per disease → baseline comparison (Logistic Regression, Random Forest, Gradient Boosting) → 5-way ablation (Full model vs. No-DFIG, No-TRE, No-Fusion, No-Shared-Trunk) → Wilcoxon signed-rank significance testing with Bonferroni correction across diseases × ablations → final full-data model trained and serialized for downstream Streamlit deployment.


## Dataset Justification & Unified Multi-Disease Methodology

The title "Multi-Disease Risk Stratification" is addressed by training on **seven** disease-specific cohorts through one shared architecture, rather than one dataset alone:

| Key | Disease | Source | Task |
|---|---|---|---|
| `cvd` | Cardiovascular disease | Cleveland Heart Disease dataset (`heart.csv`) | Binary risk classification |
| `dm` | Diabetes mellitus | BRFSS 2015 Diabetes Health Indicators (50/50 split) | Binary risk classification |
| `copd` | Chronic Obstructive Pulmonary Disease | COPD clinical dataset (`dataset.csv`) | Binary severity/risk classification |
| `alz` | Alzheimer's disease | Alzheimer's Disease clinical dataset | Binary diagnosis risk |
| `liver` | Liver disease | ILPD-style liver disease dataset | Binary diagnosis risk |
| `kidney` | Chronic kidney disease | CKD clinical dataset | Binary diagnosis risk |
| `critical` | ICU critical-illness / 30-day mortality | **MIMIC-III** (`nirmalgaud/mimic-iii-dataset` on Kaggle) — `PATIENTS`, `ADMISSIONS`, `ICUSTAYS`, `DIAGNOSES_ICD`, `LABEVENTS`, `CHARTEVENTS` tables, aggregated into one admission-level cohort table | Binary 30-day mortality risk |

**Unified dataset creation methodology:** every disease source, no matter how heterogeneous, is normalized through the same pipeline: (1) robust CSV parsing with automatic delimiter/target detection, (2) categorical encoding, (3) high-cardinality feature capping by variance, (4) median imputation, (5) binary target harmonization, and (6) age-column detection so the Temporal Risk Encoding module can locate the patient's age in each dataset automatically. The MIMIC-III branch requires additional structured-EHR feature engineering (ICU length-of-stay aggregation, lab/chart-event summary statistics, comorbidity counts from ICD-9 codes) before it enters the same pipeline as the other six flat clinical datasets — this is implemented as a dedicated cohort-builder function rather than a manual one-off script, so it is reproducible and inspectable like the rest of the pipeline.

**MIMIC-III access:** the dataset is hosted on Kaggle. This notebook first checks for a local `mimic_iii_dataset/` folder (or a `MIMIC_LOCAL_DIR` environment variable) and, if not found, downloads it automatically via `kagglehub.dataset_download("nirmalgaud/mimic-iii-dataset")`. This removes the earlier assumption that MIMIC-III would always already exist on the local machine, matching how the other six CSVs are supplied.


In [1]:
import os
import gc
import pickle
import json
import warnings
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                              average_precision_score, brier_score_loss, confusion_matrix,
                              classification_report)
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")
np.random.seed(42)

os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)

LOCAL_DATA_DIR = "."
MIMIC_LOCAL_DIR = os.environ.get("MIMIC_LOCAL_DIR", "mimic_iii_dataset")
MIMIC_KAGGLE_SLUG = "nirmalgaud/mimic-iii-dataset"

def resolve_mimic_dir():
    if os.path.isdir(MIMIC_LOCAL_DIR) and os.path.exists(os.path.join(MIMIC_LOCAL_DIR, "PATIENTS.csv")):
        return MIMIC_LOCAL_DIR
    try:
        import kagglehub
        path = kagglehub.dataset_download(MIMIC_KAGGLE_SLUG)
        for root, _, files in os.walk(path):
            if "PATIENTS.csv" in files:
                return root
        return path
    except Exception as e:
        print(f"[WARN] Could not resolve MIMIC-III directory automatically: {e}")
        print("[WARN] Set MIMIC_LOCAL_DIR env var or place MIMIC-III CSVs in ./mimic_iii_dataset")
        return None


In [2]:
MDRS_NET_LIB_SOURCE = r'''
import numpy as np


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))


def relu(x):
    return np.maximum(0, x)


def relu_grad(x):
    return (x > 0).astype(x.dtype)


class Linear:
    def __init__(self, in_dim, out_dim):
        limit = np.sqrt(6.0 / (in_dim + out_dim))
        self.W = np.random.uniform(-limit, limit, (in_dim, out_dim))
        self.b = np.zeros(out_dim)
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)
        self.x = None

    def forward(self, x):
        self.x = x
        return x @ self.W + self.b

    def backward(self, dy):
        self.dW = self.x.T @ dy
        self.db = dy.sum(axis=0)
        return dy @ self.W.T


class ReLU:
    def __init__(self):
        self.x = None

    def forward(self, x):
        self.x = x
        return relu(x)

    def backward(self, dy):
        return dy * relu_grad(self.x)


class Dropout:
    def __init__(self, p=0.2):
        self.p = p
        self.mask = None

    def forward(self, x, training=True):
        if training and self.p > 0:
            self.mask = (np.random.rand(*x.shape) > self.p) / (1 - self.p)
            return x * self.mask
        self.mask = np.ones_like(x)
        return x

    def backward(self, dy):
        return dy * self.mask


class DFIG:
    def __init__(self, in_dim):
        self.w = np.ones(in_dim) * 0.5
        self.rscale = np.array(0.3)
        self.dw = np.zeros_like(self.w)
        self.drscale = np.array(0.0)
        self.cache = None

    def forward(self, x, enabled=True):
        if not enabled:
            self.cache = ("bypass",)
            return x
        signx = np.sign(x)
        absx = np.abs(x)
        a = absx * self.w
        g = sigmoid(a)
        gated = g * x
        resid = self.rscale * self.w * x
        y = gated + resid
        self.cache = ("active", x, g, absx, signx)
        return y

    def backward(self, dy):
        mode = self.cache[0]
        if mode == "bypass":
            return dy
        _, x, g, absx, signx = self.cache
        dg_da = g * (1 - g)
        da_dx = self.w * signx
        dgated_dx = g + x * dg_da * da_dx
        dresid_dx = self.rscale * self.w
        dx = dy * (dgated_dx + dresid_dx)
        dw_total = dy * (x * dg_da * absx + self.rscale * x)
        self.dw = dw_total.sum(axis=0)
        drscale = dy * (self.w * x)
        self.drscale = drscale.sum()
        return dx

    def feature_attribution(self, x, enabled=True):
        if not enabled:
            return np.zeros_like(x)
        absx = np.abs(x)
        g = sigmoid(absx * self.w)
        gated = g * x
        resid = self.rscale * self.w * x
        contrib = gated + resid
        denom = np.sum(np.abs(contrib), axis=1, keepdims=True) + 1e-9
        return contrib / denom


class TRE:
    def __init__(self, hidden_dim=24):
        self.hidden_dim = hidden_dim
        h2 = hidden_dim // 2
        freqs = np.arange(h2).astype(np.float64)
        self.denom = 10000.0 ** (2.0 * freqs / hidden_dim)
        self.linear = Linear(hidden_dim, hidden_dim)
        self.cache = None

    def forward(self, age, enabled=True):
        B = age.shape[0]
        if not enabled:
            self.cache = ("bypass",)
            return np.zeros((B, self.hidden_dim))
        age_norm = np.clip(age, 1, 120) / 120.0
        angles = age_norm[:, None] / self.denom[None, :]
        decay = np.exp(-0.5 * (1 - age_norm))[:, None]
        sin_enc = np.sin(angles) * decay
        cos_enc = np.cos(angles) * decay
        emb_raw = np.concatenate([sin_enc, cos_enc], axis=-1)
        out = self.linear.forward(emb_raw)
        self.cache = ("active",)
        return out

    def backward(self, dy):
        if self.cache[0] == "bypass":
            return
        self.linear.backward(dy)


class CrossAttention:
    def __init__(self, dim):
        self.dim = dim
        self.q = Linear(dim, dim)
        self.k = Linear(dim, dim)
        self.v = Linear(dim, dim)
        self.scale = np.sqrt(dim)
        self.cache = None

    def forward(self, mod_a, mod_b, enabled=True):
        B = mod_a.shape[0]
        if not enabled:
            self.cache = ("bypass",)
            return np.concatenate([mod_a, mod_b], axis=-1)
        X = np.stack([mod_a, mod_b], axis=1)
        Xflat = X.reshape(B * 2, self.dim)
        Qf = self.q.forward(Xflat)
        Kf = self.k.forward(Xflat)
        Vf = self.v.forward(Xflat)
        Q = Qf.reshape(B, 2, self.dim)
        K = Kf.reshape(B, 2, self.dim)
        V = Vf.reshape(B, 2, self.dim)
        scores = np.matmul(Q, K.transpose(0, 2, 1)) / self.scale
        scores_shift = scores - scores.max(axis=-1, keepdims=True)
        expx = np.exp(scores_shift)
        A = expx / expx.sum(axis=-1, keepdims=True)
        O = np.matmul(A, V)
        out = O.reshape(B, 2 * self.dim)
        self.cache = ("active", B, Q, K, V, A)
        return out

    def backward(self, dOut):
        if self.cache[0] == "bypass":
            d = dOut.shape[-1] // 2
            return dOut[:, :d], dOut[:, d:]
        _, B, Q, K, V, A = self.cache
        dO = dOut.reshape(B, 2, self.dim)
        dA = np.matmul(dO, V.transpose(0, 2, 1))
        dV = np.matmul(A.transpose(0, 2, 1), dO)
        s = (dA * A).sum(axis=-1, keepdims=True)
        dscores = A * (dA - s)
        dscores = dscores / self.scale
        dQ = np.matmul(dscores, K)
        dK = np.matmul(dscores.transpose(0, 2, 1), Q)
        dQf = dQ.reshape(B * 2, self.dim)
        dKf = dK.reshape(B * 2, self.dim)
        dVf = dV.reshape(B * 2, self.dim)
        dXf_q = self.q.backward(dQf)
        dXf_k = self.k.backward(dKf)
        dXf_v = self.v.backward(dVf)
        dXf = dXf_q + dXf_k + dXf_v
        dX = dXf.reshape(B, 2, self.dim)
        return dX[:, 0, :], dX[:, 1, :]

    def attention_weights(self):
        if self.cache[0] == "bypass":
            return None
        return self.cache[-1]


class DiseaseBranch:
    def __init__(self, raw_dim, common_dim=64, tre_dim=24, age_idx=None):
        self.age_idx = age_idx
        self.dfig = DFIG(raw_dim)
        self.tre = TRE(hidden_dim=tre_dim)
        self.mod_a_proj = Linear(raw_dim, common_dim)
        self.mod_b_proj = Linear(tre_dim, common_dim)
        self.fusion = CrossAttention(common_dim)
        self.out_dim = common_dim * 2

    def forward(self, x, use_dfig=True, use_tre=True, use_fusion=True):
        gated = self.dfig.forward(x, enabled=use_dfig)
        if self.age_idx is not None:
            age_col = x[:, self.age_idx]
        else:
            age_col = np.full(x.shape[0], 50.0)
        tre_emb = self.tre.forward(age_col, enabled=use_tre)
        mod_a = self.mod_a_proj.forward(gated)
        mod_b = self.mod_b_proj.forward(tre_emb)
        fused = self.fusion.forward(mod_a, mod_b, enabled=use_fusion)
        return fused

    def backward(self, dfused, use_dfig=True, use_tre=True, use_fusion=True):
        dmod_a, dmod_b = self.fusion.backward(dfused)
        dgated = self.mod_a_proj.backward(dmod_a)
        dtre = self.mod_b_proj.backward(dmod_b)
        self.tre.backward(dtre)
        self.dfig.backward(dgated)


class Trunk:
    def __init__(self, in_dim, hidden=128):
        self.l1 = Linear(in_dim, hidden)
        self.r1 = ReLU()
        self.d1 = Dropout(0.25)
        self.l2 = Linear(hidden, hidden // 2)
        self.r2 = ReLU()
        self.d2 = Dropout(0.15)
        self.out_dim = hidden // 2

    def forward(self, x, training=True):
        h = self.l1.forward(x)
        h = self.r1.forward(h)
        h = self.d1.forward(h, training)
        h = self.l2.forward(h)
        h = self.r2.forward(h)
        h = self.d2.forward(h, training)
        return h

    def backward(self, dh):
        dh = self.d2.backward(dh)
        dh = self.r2.backward(dh)
        dh = self.l2.backward(dh)
        dh = self.d1.backward(dh)
        dh = self.r1.backward(dh)
        dh = self.l1.backward(dh)
        return dh


class IndividualHead:
    def __init__(self, in_dim, hidden=64):
        self.l1 = Linear(in_dim, hidden)
        self.r1 = ReLU()
        self.d1 = Dropout(0.2)
        self.l2 = Linear(hidden, 1)

    def forward(self, x, training=True):
        h = self.l1.forward(x)
        h = self.r1.forward(h)
        h = self.d1.forward(h, training)
        h = self.l2.forward(h)
        return h[:, 0]

    def backward(self, dlogit):
        dh = dlogit[:, None]
        dh = self.l2.backward(dh)
        dh = self.d1.backward(dh)
        dh = self.r1.backward(dh)
        dh = self.l1.backward(dh)
        return dh


class MDRSNetNumpy:
    def __init__(self, disease_dims, age_indices=None, common_dim=64, trunk_hidden=128, tre_dim=24, use_shared_trunk=True):
        self.use_shared_trunk = use_shared_trunk
        self.disease_dims = disease_dims
        self.common_dim = common_dim
        self.trunk_hidden = trunk_hidden
        self.tre_dim = tre_dim
        age_indices = age_indices or {}
        self.age_indices = age_indices
        self.branches = {k: DiseaseBranch(d, common_dim, tre_dim, age_indices.get(k)) for k, d in disease_dims.items()}
        branch_out = common_dim * 2
        if use_shared_trunk:
            self.trunk = Trunk(branch_out, trunk_hidden)
            self.heads = {k: Linear(self.trunk.out_dim, 1) for k in disease_dims}
        else:
            self.heads = {k: IndividualHead(branch_out) for k in disease_dims}

    def forward(self, x, key, use_dfig=True, use_tre=True, use_fusion=True, training=True):
        fused = self.branches[key].forward(x, use_dfig, use_tre, use_fusion)
        if self.use_shared_trunk:
            rep = self.trunk.forward(fused, training)
            logit = self.heads[key].forward(rep)[:, 0]
        else:
            logit = self.heads[key].forward(fused, training)
        return logit

    def backward(self, dlogit, key, use_dfig=True, use_tre=True, use_fusion=True):
        if self.use_shared_trunk:
            dh = self.heads[key].backward(dlogit[:, None])
            dfused = self.trunk.backward(dh)
        else:
            dfused = self.heads[key].backward(dlogit)
        self.branches[key].backward(dfused, use_dfig, use_tre, use_fusion)

    def collect_params(self):
        params = []

        def add(layer, names):
            for n in names:
                params.append((layer, n))

        for k, br in self.branches.items():
            add(br.dfig, ["w", "rscale"])
            add(br.tre.linear, ["W", "b"])
            add(br.mod_a_proj, ["W", "b"])
            add(br.mod_b_proj, ["W", "b"])
            add(br.fusion.q, ["W", "b"])
            add(br.fusion.k, ["W", "b"])
            add(br.fusion.v, ["W", "b"])
        if self.use_shared_trunk:
            add(self.trunk.l1, ["W", "b"])
            add(self.trunk.l2, ["W", "b"])
            for k, h in self.heads.items():
                add(h, ["W", "b"])
        else:
            for k, h in self.heads.items():
                add(h.l1, ["W", "b"])
                add(h.l2, ["W", "b"])
        return params

    def explain(self, x, key, use_dfig=True):
        return self.branches[key].dfig.feature_attribution(x, enabled=use_dfig)


class Adam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-4):
        self.lr = lr
        self.b1, self.b2 = betas
        self.eps = eps
        self.wd = weight_decay
        self.m = {}
        self.v = {}
        self.t = 0
        self.params = params

    def step(self):
        self.t += 1
        for layer, name in self.params:
            p = getattr(layer, name)
            g = getattr(layer, "d" + name) + self.wd * p
            key = (id(layer), name)
            if key not in self.m:
                self.m[key] = np.zeros_like(p)
                self.v[key] = np.zeros_like(p)
            self.m[key] = self.b1 * self.m[key] + (1 - self.b1) * g
            self.v[key] = self.b2 * self.v[key] + (1 - self.b2) * (g * g)
            mhat = self.m[key] / (1 - self.b1 ** self.t)
            vhat = self.v[key] / (1 - self.b2 ** self.t)
            new_p = p - self.lr * mhat / (np.sqrt(vhat) + self.eps)
            setattr(layer, name, new_p)


def weighted_bce_with_logits(logits, y):
    y = y.astype(np.float64)
    pos = y.sum()
    neg = len(y) - pos
    pos_weight = neg / max(pos, 1.0)
    w = np.where(y == 1, pos_weight, 1.0)
    probs = sigmoid(logits)
    eps = 1e-7
    loss = -np.mean(w * (y * np.log(probs + eps) + (1 - y) * np.log(1 - probs + eps)))
    dlogits = w * (probs - y) / len(y)
    return loss, dlogits


RISK_TIERS = [
    ("Low", 0.00, 0.25),
    ("Moderate", 0.25, 0.50),
    ("High", 0.50, 0.75),
    ("Critical", 0.75, 1.01),
]


def stratify_risk(probs):
    tiers = np.empty(len(probs), dtype=object)
    for name, lo, hi in RISK_TIERS:
        mask = (probs >= lo) & (probs < hi)
        tiers[mask] = name
    return tiers


def explain_and_stratify(model, X, key, feats, use_dfig=True, use_tre=True, use_fusion=True, top_k=5):
    logits = model.forward(X, key, use_dfig, use_tre, use_fusion, training=False)
    probs = sigmoid(logits)
    tiers = stratify_risk(probs)
    attributions = model.explain(X, key, use_dfig=use_dfig)
    reports = []
    for i in range(len(probs)):
        order = np.argsort(-np.abs(attributions[i]))[:top_k]
        top_feats = [{"feature": feats[j], "attribution": float(attributions[i, j])} for j in order]
        reports.append({
            "risk_probability": float(probs[i]),
            "risk_tier": tiers[i],
            "top_contributing_features": top_feats,
        })
    return probs, tiers, attributions, reports


def find_age_index(feats):
    for i, f in enumerate(feats):
        if f.lower() in ["age", "age_group", "age group", "years"]:
            return i
    return None
'''

with open('mdrs_net_lib.py', 'w') as _f:
    _f.write(MDRS_NET_LIB_SOURCE)

from mdrs_net_lib import *


In [3]:
TARGET_CANDIDATES = [
    "target", "Diagnosis", "diagnosis", "classification", "class",
    "Dataset", "outcome", "Outcome", "COPDSEVERITY", "Diabetes_binary",
    "status", "Status", "result", "Result", "label", "Label"
]

ID_LIKE = ["id", "patientid", "doctorincharge", "unnamed: 0"]


def robust_read_csv(path):
    df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    if df.shape[1] == 1:
        df = df[df.columns[0]].str.split(",", expand=True)
        df.columns = df.iloc[0]
        df = df.iloc[1:].reset_index(drop=True)
    df.columns = [str(c).strip().replace("\ufeff", "") for c in df.columns]
    if df.columns.duplicated().any():
        dup_cols = df.columns[df.columns.duplicated()].unique().tolist()
        print(f"[WARN] duplicate columns found and de-duplicated: {dup_cols}")
        df = df.loc[:, ~df.columns.duplicated()]
    return df


def find_target_col(df):
    col_map = {c.strip().lower(): c for c in df.columns}
    for c in TARGET_CANDIDATES:
        key = c.strip().lower()
        if key in col_map:
            return col_map[key]
    return df.columns[-1]


def auto_preprocess(df, target_col=None, max_features=40, existing_encoders=None, existing_imputer=None, existing_feats=None):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    drop_cols = [c for c in df.columns if c.lower() in ID_LIKE]
    df = df.drop(columns=drop_cols, errors="ignore")

    if target_col is None or target_col not in df.columns:
        if target_col is not None:
            print(f"[WARN] target_col '{target_col}' not found, auto-detecting instead")
        target_col = find_target_col(df)

    y_raw = df[target_col]
    if isinstance(y_raw, pd.DataFrame):
        y_raw = y_raw.iloc[:, 0]
    X_df = df.drop(columns=[target_col])

    encoders = existing_encoders or {}
    for c in X_df.columns:
        if is_numeric_dtype(X_df[c]):
            X_df[c] = pd.to_numeric(X_df[c], errors="coerce")
        else:
            if c in encoders:
                le = encoders[c]
                X_df[c] = X_df[c].astype(str).map(lambda v: v if v in le.classes_ else le.classes_[0])
                X_df[c] = le.transform(X_df[c])
            else:
                le = LabelEncoder()
                X_df[c] = le.fit_transform(X_df[c].astype(str))
                encoders[c] = le

    if existing_feats is not None:
        for c in existing_feats:
            if c not in X_df.columns:
                X_df[c] = 0.0
        X_df = X_df[existing_feats]
    elif X_df.shape[1] > max_features:
        variances = X_df.var(numeric_only=True).sort_values(ascending=False)
        keep = variances.index[:max_features]
        X_df = X_df[keep]

    if existing_imputer is not None:
        X = existing_imputer.transform(X_df)
        imputer = existing_imputer
    else:
        imputer = SimpleImputer(strategy="median")
        X = imputer.fit_transform(X_df)

    if is_numeric_dtype(y_raw):
        y_numeric = pd.to_numeric(y_raw, errors="coerce").fillna(0).values
        n_unique = len(np.unique(y_numeric))
        if n_unique > 2:
            y = (y_numeric > np.median(y_numeric)).astype(int)
        else:
            le_y = LabelEncoder()
            y = le_y.fit_transform(y_numeric)
    else:
        le_y = LabelEncoder()
        y = le_y.fit_transform(y_raw.astype(str))
        if len(np.unique(y)) > 2:
            y = (y > np.median(y)).astype(int)

    y = np.asarray(y).astype(int)
    meta = {"encoders": encoders, "imputer": imputer, "feats": list(X_df.columns), "target_col": target_col}
    return X.astype(np.float64), y, meta


In [4]:
def build_mimic_cohort(mimic_dir):
    files = {
        "PATIENTS": "PATIENTS.csv",
        "ICUSTAYS": "ICUSTAYS.csv",
        "DIAGNOSES_ICD": "DIAGNOSES_ICD.csv",
        "ADMISSIONS": "ADMISSIONS.csv",
        "LABEVENTS": "LABEVENTS.csv",
        "CHARTEVENTS": "CHARTEVENTS.csv",
    }
    dfs = {}
    for name, fname in files.items():
        path = os.path.join(mimic_dir, fname)
        if not os.path.exists(path):
            print(f"[SKIP] mimic: {fname} not found in {mimic_dir}")
            return None
        dfs[name] = pd.read_csv(path, low_memory=False)

    admissions_df = dfs["ADMISSIONS"].copy()
    patients_df = dfs["PATIENTS"].copy()
    icustays_df = dfs["ICUSTAYS"].copy()

    admissions_df["admittime"] = pd.to_datetime(admissions_df["admittime"], errors="coerce")
    admissions_df["dischtime"] = pd.to_datetime(admissions_df["dischtime"], errors="coerce")
    admissions_df["deathtime"] = pd.to_datetime(admissions_df["deathtime"], errors="coerce")
    patients_df["dod"] = pd.to_datetime(patients_df["dod"], errors="coerce")
    patients_df["dob"] = pd.to_datetime(patients_df["dob"], errors="coerce")

    cohort = admissions_df.merge(
        patients_df[["subject_id", "gender", "dob", "dod"]], on="subject_id", how="left"
    )

    cohort["age"] = cohort["admittime"].dt.year - cohort["dob"].dt.year
    cohort["age"] = cohort["age"].apply(lambda x: 90 if pd.notna(x) and x > 89 else x)

    cohort["hospital_expire_flag"] = cohort["hospital_expire_flag"].fillna(0)
    death_dates = cohort["deathtime"].fillna(cohort["dod"])
    cohort["mortality_30d"] = ((death_dates - cohort["admittime"]).dt.days <= 30).astype(int)

    icu_agg = (
        icustays_df.groupby(["subject_id", "hadm_id"])
        .agg(total_icu_los=("los", "sum"), icu_stay_count=("icustay_id", "count"),
             first_careunit=("first_careunit", "first"))
        .reset_index()
    )
    cohort = cohort.merge(icu_agg, on=["subject_id", "hadm_id"], how="left")
    cohort["total_icu_los"] = cohort["total_icu_los"].fillna(0)

    diag_df = dfs["DIAGNOSES_ICD"].copy()
    diag_counts = (
        diag_df.groupby(["subject_id", "hadm_id"]).agg(num_diagnoses=("icd9_code", "nunique")).reset_index()
    )
    cohort = cohort.merge(diag_counts, on=["subject_id", "hadm_id"], how="left")
    cohort["num_diagnoses"] = cohort["num_diagnoses"].fillna(0)

    labs_df = dfs["LABEVENTS"].copy()
    lab_stats = (
        labs_df.groupby(["subject_id", "hadm_id"])
        .agg(total_labs_count=("valuenum", "count"),
             abnormal_labs_count=("flag", lambda x: (x.astype(str).str.lower() == "abnormal").sum()),
             mean_lab_val=("valuenum", "mean"), max_lab_val=("valuenum", "max"), min_lab_val=("valuenum", "min"))
        .reset_index()
    )
    cohort = cohort.merge(lab_stats, on=["subject_id", "hadm_id"], how="left")

    chart_df = dfs["CHARTEVENTS"].copy()
    chart_stats = (
        chart_df.groupby(["subject_id", "hadm_id"])
        .agg(chart_events_count=("valuenum", "count"), mean_vital_val=("valuenum", "mean"),
             max_vital_val=("valuenum", "max"), min_vital_val=("valuenum", "min"),
             std_vital_val=("valuenum", "std"))
        .reset_index()
    )
    cohort = cohort.merge(chart_stats, on=["subject_id", "hadm_id"], how="left")

    fill_map = {
        "total_labs_count": 0, "abnormal_labs_count": 0, "chart_events_count": 0,
        "mean_lab_val": 0, "max_lab_val": 0, "min_lab_val": 0,
        "mean_vital_val": 0, "max_vital_val": 0, "min_vital_val": 0, "std_vital_val": 0, "age": 60,
    }
    cohort.fillna(fill_map, inplace=True)

    feature_cols = [
        "age", "gender", "admission_type", "insurance", "first_careunit",
        "total_icu_los", "icu_stay_count", "num_diagnoses", "total_labs_count",
        "abnormal_labs_count", "mean_lab_val", "chart_events_count", "mean_vital_val",
        "max_vital_val", "min_vital_val",
    ]

    ml_dataset = cohort[["mortality_30d"] + feature_cols].copy()
    ml_dataset = pd.get_dummies(
        ml_dataset, columns=["gender", "admission_type", "insurance", "first_careunit"], drop_first=True
    )
    ml_dataset.columns = [str(c) for c in ml_dataset.columns]
    return ml_dataset


In [5]:
DATASET_CONFIGS = {
    "cvd":    {"path": "heart.csv",   "target": "target", "label": "Cardiovascular Disease"},
    "dm":     {"path": "diabetes_binary_5050split_health_indicators_BRFSS2015.csv", "target": "Diabetes_binary", "label": "Diabetes Mellitus"},
    "copd":   {"path": "dataset.csv", "target": None, "label": "COPD"},
    "alz":    {"path": "alzheimer_disease_data.csv", "target": None, "label": "Alzheimer's Disease"},
    "liver":  {"path": "liver_disease_dataset.csv",  "target": None, "label": "Liver Disease"},
    "kidney": {"path": "kidney_disease_dataset.csv", "target": None, "label": "Chronic Kidney Disease"},
    "critical": {"path": None, "target": "mortality_30d", "label": "ICU Critical-Illness / 30-Day Mortality (MIMIC-III)"},
}


def load_all_datasets(configs=DATASET_CONFIGS, data_dir=LOCAL_DATA_DIR):
    loaded = {}
    for key, cfg in configs.items():
        if key == "critical":
            mimic_dir = resolve_mimic_dir()
            if mimic_dir is None:
                print("[SKIP] critical: MIMIC-III directory unavailable")
                continue
            df = build_mimic_cohort(mimic_dir)
            if df is None:
                print("[SKIP] critical: could not build MIMIC cohort")
                continue
        else:
            full_path = os.path.join(data_dir, cfg["path"])
            if not os.path.exists(full_path):
                print(f"[SKIP] {key}: file not found at {full_path}")
                continue
            df = robust_read_csv(full_path)

        X, y, meta = auto_preprocess(df, target_col=cfg["target"])
        loaded[key] = {"X": X, "y": y, "feats": meta["feats"], "target": meta["target_col"],
                        "encoders": meta["encoders"], "imputer": meta["imputer"], "label": cfg["label"]}
        print(f"[LOADED] {key}: n={len(y)}  features={len(meta['feats'])}  "
              f"target_col='{meta['target_col']}'  pos_rate={y.mean():.3f}")
    return loaded


## Validation Strategy

- **Benchmark datasets:** all 7 disease cohorts above.
- **Baselines:** Logistic Regression, Random Forest, Gradient Boosting (class-balanced) trained per disease under the same 5-fold protocol.
- **Evaluation metrics:** ROC-AUC, Average Precision, Brier score (calibration), F1, Precision, Recall.
- **Ablation studies:** Full MDRS-Net++ vs. No-DFIG, No-TRE, No-Cross-Attention-Fusion, No-Shared-Trunk — isolates the contribution of each proposed module.
- **Statistical significance testing:** Wilcoxon signed-rank test per disease/ablation pair, reported both at raw α = 0.05 and Bonferroni-corrected α (adjusted for the number of disease × ablation comparisons) to control for multiple-comparison inflation.


In [6]:
def train_joint_fold(train_data, disease_dims, age_indices, epochs=60, lr=1e-3,
                      use_dfig=True, use_tre=True, use_fusion=True, use_shared_trunk=True,
                      batch_size=128, steps_per_epoch=60, grad_clip=5.0):
    model = MDRSNetNumpy(disease_dims, age_indices, use_shared_trunk=use_shared_trunk)
    opt = Adam(model.collect_params(), lr=lr)
    keys = list(train_data.keys())
    for epoch in range(epochs):
        for step in range(steps_per_epoch):
            order = keys.copy()
            np.random.shuffle(order)
            for key in order:
                X, y = train_data[key]
                n = X.shape[0]
                if n >= batch_size:
                    idx = np.random.choice(n, batch_size, replace=False)
                else:
                    idx = np.random.choice(n, batch_size, replace=True)
                Xb, yb = X[idx], y[idx]
                logits = model.forward(Xb, key, use_dfig, use_tre, use_fusion, training=True)
                loss, dlogits = weighted_bce_with_logits(logits, yb)
                if grad_clip:
                    dlogits = np.clip(dlogits, -grad_clip, grad_clip)
                model.backward(dlogits, key, use_dfig, use_tre, use_fusion)
                if grad_clip:
                    for layer, pname in opt.params:
                        g = getattr(layer, "d" + pname)
                        setattr(layer, "d" + pname, np.clip(g, -grad_clip, grad_clip))
                opt.step()
    return model


def eval_model(model, X, y, key, use_dfig=True, use_tre=True, use_fusion=True):
    logits = model.forward(X, key, use_dfig, use_tre, use_fusion, training=False)
    probs = sigmoid(logits)
    preds = (probs > 0.5).astype(int)
    metrics = {}
    if len(np.unique(y)) > 1:
        metrics["AUC"] = roc_auc_score(y, probs)
        metrics["AP"] = average_precision_score(y, probs)
        metrics["Brier"] = brier_score_loss(y, probs)
    else:
        metrics["AUC"] = np.nan
        metrics["AP"] = np.nan
        metrics["Brier"] = np.nan
    metrics["F1"] = f1_score(y, preds, zero_division=0)
    metrics["Precision"] = precision_score(y, preds, zero_division=0)
    metrics["Recall"] = recall_score(y, preds, zero_division=0)
    return metrics, probs


ABLATIONS = {
    "Full_MDRS-Net":      dict(use_dfig=True,  use_tre=True,  use_fusion=True,  use_shared_trunk=True),
    "No_DFIG":            dict(use_dfig=False, use_tre=True,  use_fusion=True,  use_shared_trunk=True),
    "No_TRE":             dict(use_dfig=True,  use_tre=False, use_fusion=True,  use_shared_trunk=True),
    "No_CrossAttnFusion": dict(use_dfig=True,  use_tre=True,  use_fusion=False, use_shared_trunk=True),
    "No_SharedTrunk":     dict(use_dfig=True,  use_tre=True,  use_fusion=True,  use_shared_trunk=False),
}

BASELINE_MODELS = {
    "Logistic Regression": lambda: LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced"),
    "Random Forest": lambda: RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42, class_weight="balanced"),
    "Gradient Boosting": lambda: GradientBoostingClassifier(n_estimators=100, random_state=42),
}


def run_baselines(X, y, n_splits=5):
    min_class = np.bincount(y).min()
    splits = min(n_splits, max(2, min_class))
    skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=42)
    results = {name: [] for name in BASELINE_MODELS}
    for tr_idx, va_idx in skf.split(X, y):
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X[tr_idx])
        Xva = scaler.transform(X[va_idx])
        ytr, yva = y[tr_idx], y[va_idx]
        if len(np.unique(ytr)) < 2 or len(np.unique(yva)) < 2:
            continue
        for name, factory in BASELINE_MODELS.items():
            clf = factory()
            clf.fit(Xtr, ytr)
            p = clf.predict_proba(Xva)[:, 1]
            results[name].append(roc_auc_score(yva, p))
    return {name: (np.mean(v) if v else np.nan) for name, v in results.items()}, results


In [7]:
def cross_validate_all(loaded, n_splits=5, epochs=40, batch_size=128, steps_per_epoch=50):
    disease_dims = {k: v["X"].shape[1] for k, v in loaded.items()}
    age_indices = {k: find_age_index(v["feats"]) for k, v in loaded.items()}
    fold_indices = {}
    for key, v in loaded.items():
        min_class = np.bincount(v["y"]).min()
        splits = min(n_splits, max(2, min_class))
        skf = StratifiedKFold(n_splits=splits, shuffle=True, random_state=42)
        fold_indices[key] = list(skf.split(v["X"], v["y"]))

    n_folds = min(len(f) for f in fold_indices.values())
    ablation_fold_scores = {name: {k: [] for k in loaded} for name in ABLATIONS}

    for fold in range(n_folds):
        train_data = {}
        test_data = {}
        for key, v in loaded.items():
            tr_idx, va_idx = fold_indices[key][fold]
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(v["X"][tr_idx])
            Xva = scaler.transform(v["X"][va_idx])
            train_data[key] = (Xtr, v["y"][tr_idx])
            test_data[key] = (Xva, v["y"][va_idx])

        for ablation_name, flags in ABLATIONS.items():
            model = train_joint_fold(train_data, disease_dims, age_indices, epochs=epochs,
                                      batch_size=batch_size, steps_per_epoch=steps_per_epoch, **flags)
            for key in loaded:
                Xva, yva = test_data[key]
                metrics, _ = eval_model(model, Xva, yva, key,
                                         use_dfig=flags["use_dfig"], use_tre=flags["use_tre"],
                                         use_fusion=flags["use_fusion"])
                ablation_fold_scores[ablation_name][key].append(metrics)
            print(f"[Fold {fold+1}/{n_folds}] {ablation_name} done")
            del model
            gc.collect()

        del train_data, test_data
        gc.collect()

    return ablation_fold_scores


def summarize_and_test(ablation_fold_scores, baseline_results, loaded):
    rows = []
    for ablation_name, per_disease in ablation_fold_scores.items():
        for disease, fold_metrics in per_disease.items():
            aucs = [m["AUC"] for m in fold_metrics if not np.isnan(m["AUC"])]
            f1s = [m["F1"] for m in fold_metrics]
            briers = [m["Brier"] for m in fold_metrics if not np.isnan(m["Brier"])]
            rows.append({
                "Model": ablation_name, "Disease": disease,
                "AUC_mean": np.mean(aucs) if aucs else np.nan,
                "AUC_std": np.std(aucs) if aucs else np.nan,
                "F1_mean": np.mean(f1s), "F1_std": np.std(f1s),
                "Brier_mean": np.mean(briers) if briers else np.nan,
            })
    summary_df = pd.DataFrame(rows)

    sig_rows = []
    n_diseases = len(loaded)
    n_ablation_comparisons = len(ABLATIONS) - 1
    bonferroni_alpha = 0.05 / max(1, n_diseases * n_ablation_comparisons)
    full_aucs = {d: [m["AUC"] for m in ablation_fold_scores["Full_MDRS-Net"][d]] for d in loaded}
    for ablation_name in ABLATIONS:
        if ablation_name == "Full_MDRS-Net":
            continue
        for disease in loaded:
            other_aucs = [m["AUC"] for m in ablation_fold_scores[ablation_name][disease]]
            fa = [a for a in full_aucs[disease] if not np.isnan(a)]
            oa = [a for a in other_aucs if not np.isnan(a)]
            n = min(len(fa), len(oa))
            if n >= 5 and not np.allclose(fa[:n], oa[:n]):
                try:
                    stat, p = wilcoxon(fa[:n], oa[:n])
                except ValueError:
                    stat, p = np.nan, np.nan
            else:
                stat, p = np.nan, np.nan
            sig_rows.append({
                "Comparison": f"Full_MDRS-Net vs {ablation_name}",
                "Disease": disease, "Wilcoxon_stat": stat, "p_value": p,
                "Significant_p<0.05": (p < 0.05) if not np.isnan(p) else False,
                "Significant_Bonferroni": (p < bonferroni_alpha) if not np.isnan(p) else False,
            })
    sig_df = pd.DataFrame(sig_rows)
    sig_df.attrs["bonferroni_alpha"] = bonferroni_alpha

    baseline_rows = []
    for disease, bmeans in baseline_results.items():
        for bname, bmean in bmeans.items():
            baseline_rows.append({"Disease": disease, "Model": bname, "AUC_mean": bmean})
    for _, row in summary_df[summary_df["Model"] == "Full_MDRS-Net"].iterrows():
        baseline_rows.append({"Disease": row["Disease"], "Model": "MDRS-Net (Ours)", "AUC_mean": row["AUC_mean"]})
    baseline_df = pd.DataFrame(baseline_rows)

    summary_df.to_csv("results/ablation_summary.csv", index=False)
    sig_df.to_csv("results/significance_tests.csv", index=False)
    baseline_df.to_csv("results/baseline_comparison.csv", index=False)
    return summary_df, sig_df, baseline_df


In [8]:
def train_final_deployment_model(loaded, epochs=60, batch_size=128, steps_per_epoch=60, lr=1e-3):
    disease_dims = {k: v["X"].shape[1] for k, v in loaded.items()}
    age_indices = {k: find_age_index(v["feats"]) for k, v in loaded.items()}
    scalers = {}
    train_data = {}
    for key, v in loaded.items():
        scaler = StandardScaler()
        Xs = scaler.fit_transform(v["X"])
        scalers[key] = scaler
        train_data[key] = (Xs, v["y"])

    model = train_joint_fold(train_data, disease_dims, age_indices, epochs=epochs,
                              batch_size=batch_size, steps_per_epoch=steps_per_epoch, lr=lr,
                              use_dfig=True, use_tre=True, use_fusion=True, use_shared_trunk=True)

    final_metrics = {}
    for key, v in loaded.items():
        Xs, y = train_data[key]
        metrics, probs = eval_model(model, Xs, y, key)
        tiers = stratify_risk(probs)
        tier_counts = pd.Series(tiers).value_counts().to_dict()
        final_metrics[key] = {**metrics, "risk_tier_distribution": tier_counts}
        print(f"[FINAL] {key}: AUC={metrics['AUC']:.4f} F1={metrics['F1']:.4f} "
              f"Precision={metrics['Precision']:.4f} Recall={metrics['Recall']:.4f}")

    deployment_bundle = {
        "model": model,
        "scalers": scalers,
        "disease_dims": disease_dims,
        "age_indices": age_indices,
        "feats": {k: v["feats"] for k, v in loaded.items()},
        "encoders": {k: v["encoders"] for k, v in loaded.items()},
        "imputer": {k: v["imputer"] for k, v in loaded.items()},
        "labels": {k: v["label"] for k, v in loaded.items()},
        "risk_tiers": RISK_TIERS,
        "final_metrics": final_metrics,
        "ablations_used": ABLATIONS["Full_MDRS-Net"],
    }

    with open("models/mdrs_net_final_model.pkl", "wb") as f:
        pickle.dump(deployment_bundle, f)

    with open("models/mdrs_net_metadata.json", "w") as f:
        json.dump({
            "diseases": list(loaded.keys()),
            "labels": {k: v["label"] for k, v in loaded.items()},
            "n_features": {k: v["X"].shape[1] for k, v in loaded.items()},
            "feature_names": {k: v["feats"] for k, v in loaded.items()},
            "risk_tiers": [{"name": n, "low": lo, "high": hi} for n, lo, hi in RISK_TIERS],
            "final_metrics": {k: {mk: (mv if not isinstance(mv, dict) else mv) for mk, mv in m.items() if mk != "risk_tier_distribution"} for k, m in final_metrics.items()},
        }, f, indent=2, default=str)

    print("\n[SAVED] models/mdrs_net_final_model.pkl")
    print("[SAVED] models/mdrs_net_metadata.json")
    return model, deployment_bundle, final_metrics


In [9]:
def main():
    loaded = load_all_datasets()
    if not loaded:
        print("No datasets found. Place the 6 CSVs in the working directory, "
              "and set MIMIC_LOCAL_DIR or ensure kagglehub access for MIMIC-III, then rerun.")
        return None

    baseline_results = {}
    for key, v in loaded.items():
        means, _ = run_baselines(v["X"], v["y"])
        baseline_results[key] = means
        print(f"[Baselines] {key}: {means}")

    ablation_fold_scores = cross_validate_all(loaded, n_splits=5, epochs=40, batch_size=128, steps_per_epoch=50)
    summary_df, sig_df, baseline_df = summarize_and_test(ablation_fold_scores, baseline_results, loaded)

    print("\nABLATION SUMMARY")
    print(summary_df.to_string(index=False))
    print("\nSIGNIFICANCE TESTS (Full vs each ablation, Bonferroni alpha={:.5f})".format(
        sig_df.attrs.get("bonferroni_alpha", 0.05)))
    print(sig_df.to_string(index=False))
    print("\nBASELINE COMPARISON (incl. MDRS-Net)")
    print(baseline_df.to_string(index=False))
    print("\nSaved: results/ablation_summary.csv, results/significance_tests.csv, results/baseline_comparison.csv")

    model, bundle, final_metrics = train_final_deployment_model(loaded, epochs=60, batch_size=128, steps_per_epoch=60, lr=1e-3)

    return {
        "loaded": loaded,
        "baseline_results": baseline_results,
        "ablation_fold_scores": ablation_fold_scores,
        "summary_df": summary_df,
        "sig_df": sig_df,
        "baseline_df": baseline_df,
        "final_model": model,
        "deployment_bundle": bundle,
        "final_metrics": final_metrics,
    }


results_bundle = main()


[WARN] duplicate columns found and de-duplicated: ['0', '1', '2']
[WARN] target_col 'target' not found, auto-detecting instead
[LOADED] cvd: n=1024  features=7  target_col='3'  pos_rate=0.399
[LOADED] dm: n=70692  features=21  target_col='Diabetes_binary'  pos_rate=0.500
[WARN] duplicate columns found and de-duplicated: ['120', '1', '0']
[LOADED] copd: n=100  features=16  target_col='0'  pos_rate=0.190
[LOADED] alz: n=2149  features=32  target_col='Diagnosis'  pos_rate=0.354
[LOADED] liver: n=583  features=10  target_col='Dataset'  pos_rate=0.286
[LOADED] kidney: n=400  features=24  target_col='classification'  pos_rate=0.380
[LOADED] critical: n=129  features=21  target_col='mortality_30d'  pos_rate=0.372
[Baselines] cvd: {'Logistic Regression': 0.7628564713831419, 'Random Forest': 0.9997620464009518, 'Gradient Boosting': 0.9582986317668054}
[Baselines] dm: {'Logistic Regression': 0.824675252816124, 'Random Forest': 0.8094631597296302, 'Gradient Boosting': 0.8307312287723179}
[Baselin

## Expected Scientific Contributions

1. A reusable, disease-agnostic multi-task architecture (MDRS-Net++) with three custom-built modules (DFIG, TRE, Cross-Modal Attention Fusion) implemented and differentiated from first principles.
2. An architecture-native explainability layer (XRS) that produces per-patient risk tiers and feature attributions without a separate post-hoc explainer.
3. A unified data-engineering methodology that harmonizes six flat clinical datasets and one structured EHR database (MIMIC-III) into a single multi-disease training protocol.
4. A rigorous validation protocol combining baseline comparison, ablation analysis, and multiple-comparison-corrected significance testing, directly supporting claims of methodological contribution rather than integration alone.
5. A serialized, deployment-ready model bundle (architecture + weights + scalers + encoders + risk-tier definitions) enabling immediate reuse in a downstream Streamlit clinical risk-stratification application.


In [10]:
import glob
from IPython.display import display

for file in sorted(glob.glob(os.path.join("results", "*.csv"))):
    print(f"### {os.path.basename(file)}")
    df = pd.read_csv(file)
    display(df)
    print(f"Shape: {df.shape}")
    print("-" * 100)

print("\nDeployment model saved at: models/mdrs_net_final_model.pkl")
print("Deployment metadata saved at: models/mdrs_net_metadata.json")
print("Model architecture module (required for loading the pickle elsewhere, e.g. Streamlit): mdrs_net_lib.py")


### ablation_summary.csv


,Model,Disease,AUC_mean,AUC_std,F1_mean,F1_std,Brier_mean
0,Full_MDRS-Net,cvd,0.985651,0.012055,0.921885,0.030534,0.047903
1,Full_MDRS-Net,dm,0.824964,0.002915,0.761629,0.003480,0.170183
2,Full_MDRS-Net,copd,0.651900,0.165912,0.341587,0.263669,0.239680
3,Full_MDRS-Net,alz,0.903932,0.021739,0.788805,0.022013,0.118235
4,Full_MDRS-Net,liver,0.738361,0.035532,0.499961,0.083400,0.244170
5,Full_MDRS-Net,kidney,0.991942,0.007007,0.964901,0.017364,0.026315
6,Full_MDRS-Net,critical,0.574056,0.082009,0.437624,0.104776,0.420604
7,No_DFIG,cvd,0.982402,0.018978,0.912225,0.055776,0.048696
8,No_DFIG,dm,0.817593,0.008931,0.757829,0.006457,0.173227
9,No_DFIG,copd,0.618382,0.135952,0.400000,0.275681,0.235292


Shape: (35, 7)
----------------------------------------------------------------------------------------------------
### baseline_comparison.csv


,Disease,Model,AUC_mean
0,cvd,Logistic Regression,0.762856
1,cvd,Random Forest,0.999762
2,cvd,Gradient Boosting,0.958299
3,dm,Logistic Regression,0.824675
4,dm,Random Forest,0.809463
5,dm,Gradient Boosting,0.830731
6,copd,Logistic Regression,0.419792
7,copd,Random Forest,0.568689
8,copd,Gradient Boosting,0.572243
9,alz,Logistic Regression,0.897918


Shape: (28, 3)
----------------------------------------------------------------------------------------------------
### cv_results.csv


,Disease,AUC,AP,F1,Precision,Recall
0,CVD,0.986571,0.986811,0.956938,0.961538,0.952381
1,CVD,0.993714,0.995993,0.985782,0.981132,0.990476
2,CVD,0.998857,0.998970,0.985507,1.000000,0.971429
3,CVD,0.995810,0.996388,0.965517,1.000000,0.933333
4,CVD,0.991995,0.991669,0.966507,0.980583,0.952830
5,DM,0.824055,0.794640,0.756783,0.734023,0.781000
6,DM,0.813450,0.786315,0.746261,0.732083,0.761000
7,DM,0.828846,0.805249,0.767364,0.731851,0.806500
8,DM,0.826659,0.801077,0.752204,0.737044,0.768000
9,DM,0.819210,0.785721,0.760009,0.722197,0.802000


Shape: (13, 6)
----------------------------------------------------------------------------------------------------
### significance_tests.csv


,Comparison,Disease,Wilcoxon_stat,p_value,Significant_p<0.05,Significant_Bonferroni
0,Full_MDRS-Net vs No_DFIG,cvd,6.0,0.81250,False,False
1,Full_MDRS-Net vs No_DFIG,dm,0.0,0.06250,False,False
2,Full_MDRS-Net vs No_DFIG,copd,6.0,0.81250,False,False
3,Full_MDRS-Net vs No_DFIG,alz,3.0,0.31250,False,False
4,Full_MDRS-Net vs No_DFIG,liver,1.0,0.12500,False,False
5,Full_MDRS-Net vs No_DFIG,kidney,2.0,0.59298,False,False
6,Full_MDRS-Net vs No_DFIG,critical,2.0,0.18750,False,False
7,Full_MDRS-Net vs No_TRE,cvd,3.0,0.31250,False,False
8,Full_MDRS-Net vs No_TRE,dm,5.0,0.62500,False,False
9,Full_MDRS-Net vs No_TRE,copd,7.0,1.00000,False,False


Shape: (28, 6)
----------------------------------------------------------------------------------------------------

Deployment model saved at: models/mdrs_net_final_model.pkl
Deployment metadata saved at: models/mdrs_net_metadata.json
Model architecture module (required for loading the pickle elsewhere, e.g. Streamlit): mdrs_net_lib.py
